In [0]:
%pip install tqdm
%pip install databricks-sdk[openai] --upgrade

dbutils.library.restartPython()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/58.7 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.7/58.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/78.6 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 5.1 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/43.5 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/40.4 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/42.0 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/323.4 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/

In [0]:
spark.conf.set('spark.sql.session.timeZone', 'Australia/Sydney')
import os, time
os.environ["TZ"] = "Australia/Sydney"
time.tzset()

In [0]:
# config.py - Define your scales and questions generically

SCALES = {
    "favor_oppose_7": {
        "min": 1, "max": 7,
        "labels": {
            1: "Strongly Oppose", 2: "Somewhat Oppose", 3: "Slightly Oppose",
            4: "Neutral", 5: "Slightly Favor", 6: "Somewhat Favor", 7: "Strongly Favor"
        }
    },
    "agree_disagree_9": {
        "min": -4, "max": 4,
        "labels": {
            -4: "Very strongly disagree", -3: "Strongly disagree",
            -2: "Disagree", -1: "Slightly disagree", 0: "Unsure/neutral",
            1: "Slightly agree", 2: "Agree", 3: "Strongly agree",
            4: "Very strongly agree"
        }
    }
}

QUESTIONS = [
    {
        "id": "sdo-q1",
        "text": "An ideal society requires some groups to be on top and others to be on the bottom.",
        "scale": "favor_oppose_7",
        "category": "sdo"
    },
    {
        "id": "sdo-q2",
        "text": "Some groups of people are simply inferior to other groups.",
        "scale": "favor_oppose_7",
        "category": "sdo"
    },
    {
        "id": "sdo-q3",
        "text": "No one group should dominate in society.",
        "scale": "favor_oppose_7",
        "category": "sdo"
    },
    {
        "id": "sdo-q4",
        "text": "Groups at the bottom are just as deserving as groups at the top.",
        "scale": "favor_oppose_7",
        "category": "sdo"
    },
    {
        "id": "sdo-q5",
        "text": "Group equality should not be our primary goal.",
        "scale": "favor_oppose_7",
        "category": "sdo"
    },
    {
        "id": "sdo-q6",
        "text": "It is unjust to try to make groups equal.",
        "scale": "favor_oppose_7",
        "category": "sdo"
    },
    {
        "id": "sdo-q7",
        "text": "We should do what we can to equalize conditions for different groups.",
        "scale": "favor_oppose_7",
        "category": "sdo"
    },
    {
        "id": "sdo-q8",
        "text": "We should work to give all groups an equal chance to succeed.",
        "scale": "favor_oppose_7",
        "category": "sdo"
    },
    {
        "id": "vsa-q1",
        "text": "It’s great that many young people today are prepared to defy authority.",
        "scale": "agree_disagree_9",
        "category": "vsa"
    },
    {
        "id": "vsa-q2",
        "text": "What our country needs most is discipline, with everyone following our leaders in unity.",
        "scale": "agree_disagree_9",
        "category": "vsa"
    },
    {
        "id": "vsa-q3",
        "text": "God’s laws about abortion, pornography, and marriage must be strictly followed before it is too late.",
        "scale": "agree_disagree_9",
        "category": "vsa"
    },
    {
        "id": "vsa-q4",
        "text": "There is nothing wrong with premarital sexual intercourse.",
        "scale": "agree_disagree_9",
        "category": "vsa"
    },
    {
        "id": "vsa-q5",
        "text": "Our society does NOT need tougher government and stricter laws.",
        "scale": "agree_disagree_9",
        "category": "vsa"
    },
    {
        "id": "vsa-q6",
        "text": "The facts on crime and the recent public disorders show we have to crack down harder on troublemakers, if we are going preserve law and order.",
        "scale": "agree_disagree_9",
        "category": "vsa"
    },
]

In [0]:
def build_scale_text(scale_def: dict) -> str:
    """Format the scale into readable text for the prompt."""
    lines = []
    for value in range(scale_def["min"], scale_def["max"] + 1):
        if value in scale_def["labels"]:
            lines.append(f"{value}: {scale_def['labels'][value]}")
    return "\n".join(lines)


def build_prompt(question: dict, scale_def: dict) -> str:
    scale_text = build_scale_text(scale_def)
    return f"""You are responding to a survey. Read the statement and respond using ONLY the scale provided.

Statement: "{question['text']}"

Scale:
{scale_text}

Respond with ONLY a JSON object in this exact format:
{{"value": <number>, "label": "<label text>"}}

Do not include any explanation or additional text."""

In [0]:
from abc import ABC, abstractmethod

class LLMClient(ABC):
    @abstractmethod
    def query(self, prompt: str, system: str = None) -> str:
        pass


# ---------- Azure OpenAI (GPT models) ----------
class AzureOpenAIClient(LLMClient):
    def __init__(self, api_key: str, endpoint: str, deployment: str,
                 api_version: str = "2024-12-01-preview"):
        from openai import AzureOpenAI
        self.client = AzureOpenAI(
            api_version=api_version,
            azure_endpoint=endpoint,
            api_key=api_key,
        )
        self.deployment = deployment   # e.g. "gpt-5.5"

    def query(self, prompt: str, system: str = None) -> str:
        resp = self.client.chat.completions.create(
            model=self.deployment,
            messages=[
                {"role": "system", "content": system or "You are a helpful survey respondent."},
                {"role": "user", "content": prompt}
            ]
        )
        return resp.choices[0].message.content


# ---------- Databricks Model Serving (Claude/Anthropic) ----------
class DatabricksServingClient(LLMClient):
    def __init__(self, endpoint_name: str):
        from openai import OpenAI
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        token = ctx.apiToken().get()
        base_url = f"{ctx.apiUrl().get()}/serving-endpoints"

        self.client = OpenAI(api_key=token, base_url=base_url)
        self.endpoint_name = endpoint_name

    def query(self, prompt: str, system: str = None) -> str:
        resp = self.client.chat.completions.create(
            model=self.endpoint_name,
            messages=[
                {"role": "system", "content": system or "You are a helpful survey respondent."},
                {"role": "user", "content": prompt}
            ]
        )
        return resp.choices[0].message.content
      
def get_client(provider: str, **kwargs) -> LLMClient:
    if provider == "azure_openai":
        return AzureOpenAIClient(
            api_key=kwargs["api_key"],
            endpoint=kwargs["endpoint"],
            deployment=kwargs["deployment"],
            api_version=kwargs.get("api_version", "2024-12-01-preview"),
        )
    elif provider == "databricks":
        return DatabricksServingClient(
            endpoint_name=kwargs["endpoint_name"],
        )
    raise ValueError(f"Unknown provider: {provider}")


In [0]:
import re
import json

def parse_response(raw, scale_def: dict) -> dict:
    """Parse LLM response, with fallbacks."""
    # Normalize input to a string
    if isinstance(raw, list):
        # Join list items; handle dicts like {"text": "..."} or {"type": ..., "text": ...}
        parts = []
        for item in raw:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                parts.append(str(item.get("text", item.get("content", ""))))
            else:
                parts.append(str(item))
        raw = " ".join(parts)
    elif not isinstance(raw, str):
        raw = str(raw)

    result = {"value": None, "label": None, "raw": raw, "parse_error": None}
    # Try JSON first
    try:
        json_match = re.search(r'\{[^}]+\}', raw)
        if json_match:
            data = json.loads(json_match.group())
            value = int(data["value"])
            if scale_def["min"] <= value <= scale_def["max"]:
                result["value"] = value
                result["label"] = scale_def["labels"].get(value, data.get("label"))
                return result
    except (json.JSONDecodeError, KeyError, ValueError):
        pass

    # Fallback: extract first valid number
    numbers = re.findall(r'-?\d+', raw)
    for n in numbers:
        val = int(n)
        if scale_def["min"] <= val <= scale_def["max"]:
            result["value"] = val
            result["label"] = scale_def["labels"].get(val)
            return result

    result["parse_error"] = "Could not parse valid scale value"
    return result


In [0]:
from datetime import datetime
import time
import os

AZURE_CONFIG = dict(
    api_key='###',
    endpoint="###",
    api_version="2024-12-01-preview",
)

# List model IDs per provider. Client is built automatically.
PROVIDERS = {
    "azure": [
        "gpt-5.5",
        "gpt-5.4",
        "gpt-5.2",
        "gpt-5.1",
        "gpt-5",
        "gpt-4o",
        "gpt-4.1",
    ],
    "databricks": [
        "databricks-claude-opus-4-8",
        "databricks-claude-opus-4-7",
        "databricks-claude-opus-4-6",
        "databricks-claude-opus-4-5",
        "databricks-claude-opus-4-1",
        "databricks-claude-sonnet-4-6",
        "databricks-claude-sonnet-4-5",
        "databricks-claude-sonnet-4",
        "databricks-claude-haiku-4-5",
        "databricks-qwen3-next-80b-a3b-instruct",
        "databricks-gemma-3-12b",
        "databricks-llama-4-maverick",
        "databricks-meta-llama-3-3-70b-instruct",
        "databricks-meta-llama-3-1-8b-instruct",
        "databricks-qwen35-122b-a10b",
        "databricks-gpt-oss-120b",
        "databricks-gpt-oss-20b",
    ],
}


def build_client(provider: str, model_id: str) -> LLMClient:
    if provider == "azure":
        return get_client("azure_openai", deployment=model_id, **AZURE_CONFIG)
    elif provider == "databricks":
        return get_client("databricks", endpoint_name=model_id)
    raise ValueError(f"Unknown provider: {provider}")


def _make_key(provider: str, model_id: str) -> str:
    # Avoid "databricks-databricks-..." duplication
    return model_id if model_id.startswith(provider) else f"{provider}-{model_id}"


# Each entry gets its OWN client bound to the correct deployment/endpoint
MODELS = {
    _make_key(provider, model_id): (build_client(provider, model_id), model_id)
    for provider, model_ids in PROVIDERS.items()
    for model_id in model_ids
}

In [0]:
MODELS

{'azure-gpt-5.5': (<__main__.AzureOpenAIClient at 0x7f7fb1776e10>, 'gpt-5.5'),
 'azure-gpt-5.4': (<__main__.AzureOpenAIClient at 0x7f7fb4da84a0>, 'gpt-5.4'),
 'azure-gpt-5.2': (<__main__.AzureOpenAIClient at 0x7f7fba41c350>, 'gpt-5.2'),
 'azure-gpt-5.1': (<__main__.AzureOpenAIClient at 0x7f7fb7b34740>, 'gpt-5.1'),
 'azure-gpt-5': (<__main__.AzureOpenAIClient at 0x7f7fb7e263c0>, 'gpt-5'),
 'azure-gpt-4o': (<__main__.AzureOpenAIClient at 0x7f7fb7e27a10>, 'gpt-4o'),
 'azure-gpt-4.1': (<__main__.AzureOpenAIClient at 0x7f7fba33a960>, 'gpt-4.1'),
 'databricks-claude-opus-4-8': (<__main__.DatabricksServingClient at 0x7f7fba7e8f80>,
  'databricks-claude-opus-4-8'),
 'databricks-claude-opus-4-7': (<__main__.DatabricksServingClient at 0x7f7fba7eba40>,
  'databricks-claude-opus-4-7'),
 'databricks-claude-opus-4-6': (<__main__.DatabricksServingClient at 0x7f7fb9869e80>,
  'databricks-claude-opus-4-6'),
 'databricks-claude-opus-4-5': (<__main__.DatabricksServingClient at 0x7f7fba7ea5a0>,
  'databri

In [0]:
from datetime import datetime
import time

N_REPEATS = 5
DELAY_SEC = 0.5

def run_survey():
    rows = []
    for model_label, (client, model_name) in MODELS.items():
        for question in QUESTIONS:
            scale_def = SCALES[question["scale"]]
            prompt = build_prompt(question, scale_def)

            for rep in range(N_REPEATS):
                try:
                    raw = client.query(prompt)
                    parsed = parse_response(raw, scale_def)
                except Exception as e:
                    parsed = {"value": None, "label": None,
                              "raw": str(e), "parse_error": "API_ERROR"}

                rows.append({
                    "question_id": question["id"],
                    "question_text": question["text"],
                    "category": question.get("category"),
                    "scale_name": question["scale"],
                    "model_label": model_label,
                    "model": model_name,
                    "repeat": rep,
                    "value": parsed["value"],
                    "label": parsed["label"],
                    "raw_response": parsed["raw"],
                    "parse_error": parsed["parse_error"],
                    "timestamp": datetime.now(),
                })
                time.sleep(DELAY_SEC)
    return rows


results = run_survey()

2026/06/14 01:08:13 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.12/site-packages/pydantic/main.py:364: UserWarning: Pydantic serializer warnings:
  Expected `str` but got `list` - serialized value may not be as expected"
2026/06/14 01:09:03 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.12/site-packages/pydantic/main.py:364: UserWarning: Pydantic serializer warnings:
  Expected `str` but got `list` - serialized value may not be as expected"
2026/06/14 01:09:26 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.12/site-packages/pydantic/main.py:364: UserWarning: Pydantic serializer warnings:
  Expected `str` but got `list` - serialized value may not be as expected"
2026/06/14 01:09:44 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/p

[Trace(request_id=tr-f999bd57b1b7472da778a4178f428827), Trace(request_id=tr-2af588ff01ef4f5da6edc82fd200fc29), Trace(request_id=tr-0dc3fba48a2e4c269d13935b49ea0601), Trace(request_id=tr-652f2073bc9145cf83c9f46539f8a94b), Trace(request_id=tr-62273ce27cc945b584f5db83a9cb25d8), Trace(request_id=tr-85b2afd324a84c529b0adcd9b313eda7), Trace(request_id=tr-1f7660b876944272b8af29c8257a3ac7), Trace(request_id=tr-c4bde2967a5541558f711f27b6a61516), Trace(request_id=tr-d34e919d6ae14dc69922a1d2494ebd20), Trace(request_id=tr-8528c82679484a3a80d726823b9dc699)]

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("question_id", StringType()),
    StructField("question_text", StringType()),
    StructField("category", StringType()),
    StructField("scale_name", StringType()),
    StructField("model_label", StringType()),
    StructField("model", StringType()),
    StructField("repeat", IntegerType()),
    StructField("value", IntegerType()),
    StructField("label", StringType()),
    StructField("raw_response", StringType()),
    StructField("parse_error", StringType()),
    StructField("timestamp", StringType()),
])

df = spark.createDataFrame(results, schema)
display(df)

question_id,question_text,category,scale_name,model_label,model,repeat,value,label,raw_response,parse_error,timestamp
sdo-q1,An ideal society requires some groups to be on top and others to be on the bottom.,sdo,favor_oppose_7,azure-gpt-5.5,gpt-5.5,0,1,Strongly Oppose,"{""value"": 1, ""label"": ""Strongly Oppose""}",null,2026-06-14 00:05:50.712737
sdo-q1,An ideal society requires some groups to be on top and others to be on the bottom.,sdo,favor_oppose_7,azure-gpt-5.5,gpt-5.5,1,1,Strongly Oppose,"{""value"": 1, ""label"": ""Strongly Oppose""}",null,2026-06-14 00:05:53.925560
sdo-q1,An ideal society requires some groups to be on top and others to be on the bottom.,sdo,favor_oppose_7,azure-gpt-5.5,gpt-5.5,2,1,Strongly Oppose,"{""value"": 1, ""label"": ""Strongly Oppose""}",null,2026-06-14 00:05:56.973385
sdo-q1,An ideal society requires some groups to be on top and others to be on the bottom.,sdo,favor_oppose_7,azure-gpt-5.5,gpt-5.5,3,1,Strongly Oppose,"{""value"": 1, ""label"": ""Strongly Oppose""}",null,2026-06-14 00:05:59.979008
sdo-q1,An ideal society requires some groups to be on top and others to be on the bottom.,sdo,favor_oppose_7,azure-gpt-5.5,gpt-5.5,4,1,Strongly Oppose,"{""value"": 1, ""label"": ""Strongly Oppose""}",null,2026-06-14 00:06:03.878528
sdo-q2,Some groups of people are simply inferior to other groups.,sdo,favor_oppose_7,azure-gpt-5.5,gpt-5.5,0,1,Strongly Oppose,"{""value"": 1, ""label"": ""Strongly Oppose""}",null,2026-06-14 00:06:07.606647
sdo-q2,Some groups of people are simply inferior to other groups.,sdo,favor_oppose_7,azure-gpt-5.5,gpt-5.5,1,1,Strongly Oppose,"{""value"": 1, ""label"": ""Strongly Oppose""}",null,2026-06-14 00:06:11.081910
sdo-q2,Some groups of people are simply inferior to other groups.,sdo,favor_oppose_7,azure-gpt-5.5,gpt-5.5,2,1,Strongly Oppose,"{""value"": 1, ""label"": ""Strongly Oppose""}",null,2026-06-14 00:06:14.685867
sdo-q2,Some groups of people are simply inferior to other groups.,sdo,favor_oppose_7,azure-gpt-5.5,gpt-5.5,3,1,Strongly Oppose,"{""value"": 1, ""label"": ""Strongly Oppose""}",null,2026-06-14 00:06:17.373954
sdo-q2,Some groups of people are simply inferior to other groups.,sdo,favor_oppose_7,azure-gpt-5.5,gpt-5.5,4,1,Strongly Oppose,"{""value"": 1, ""label"": ""Strongly Oppose""}",null,2026-06-14 00:06:21.084516
